# CGM IC Snapshot

This notebook assembles the float-native fields needed for initial-condition generation.


In [ ]:
import sys
from pathlib import Path

def _find_repo_root():
    for start in (Path.cwd(), Path.cwd().resolve()):
        for candidate in (start, *start.parents):
            if (candidate / 'pysrc' / 'solve_ode.py').exists():
                return candidate
    raise RuntimeError('Could not find repo root containing pysrc/solve_ode.py')

repo_root = _find_repo_root()
pysrc_path = repo_root / 'pysrc'
if str(pysrc_path) not in sys.path:
    sys.path.insert(0, str(pysrc_path))

import numpy as np

import solve_ode as CF
import HaloPotential_new as Halo
import WiersmaCooling as Cool
from cosmology import DEFAULT_COSMOLOGY


In [ ]:
rho_mean = DEFAULT_COSMOLOGY.mean_matter_density_Msun_kpc3(0.0)
potential = Halo.CombinedPotential(
    M_vir_Msun=1.2e11,
    r_vir_kpc=115.0,
    c_vir=10.0,
    M_gal_Msun=1.1e10,
    a_gal_kpc=3.2,
    rho_mean_Msun_kpc3=rho_mean,
    R200_kpc=145.0,
)
disk = Halo.MiyamotoNagaiPotential(M_Msun=1.1e10, a_kpc=3.2, b_kpc=0.4)
cooling = Cool.Constant_Cooling(1.0e-22)
solution = CF.IntegrateFlowEquations(
    mass_flow_rate_Msun_per_yr=0.02,
    temperature_K=3.0e5,
    density_cgs=1.5e-27,
    potential=potential,
    cooling=cooling,
    direction=1,
    R_min_kpc=20.0,
    R_max_kpc=80.0,
)
ic_snapshot = {
    'radius_kpc': solution.radius_kpc[:5].tolist(),
    'density_cgs': solution.density_cgs[:5].tolist(),
    'temperature_K': solution.temperature_K[:5].tolist(),
    'velocity_kms': solution.velocity_kms[:5].tolist(),
    'disk_phi_midplane_kms2': float(disk.phi_kms2(8.0, 0.0)),
}
ic_snapshot
